# 11.1 文档处理 (Document Processing)

RAG系统的第一步是将原始文档处理成可检索的单元。文档处理的质量直接影响检索精度和最终生成质量。

本节涵盖：
- 文本分块策略
- 递归字符分块
- 元数据提取
- 向量化与索引构建
- 工业级文档处理流水线

## 1. 文本分块策略

**目的**：将长文档切分为合适大小的片段，平衡信息完整性和检索精度

**分块大小的影响**：
- **太大**（>1000 tokens）：信息不聚焦，检索噪声大，LLM输入浪费
- **太小**（<100 tokens）：缺乏上下文，语义不完整
- **推荐**：256-512 tokens，overlap 10-20%

**分块方法对比**：
| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 固定大小 | 简单可控 | 可能切断语义 | 通用基线 |
| 句子级 | 语义完整 | 片段可能太短 | 短文档 |
| 段落级 | 语义完整 | 片段大小不均 | 结构化文档 |
| 递归字符 | 灵活自适应 | 需要调参 | 生产环境 |
| 语义分块 | 最优语义边界 | 计算成本高 | 高精度需求 |

In [ ]:
import re

text = """Machine learning is a subset of artificial intelligence that enables systems to learn from data. It has revolutionized many industries including healthcare, finance, and technology.

Deep learning uses neural networks with multiple layers. It excels at tasks like image recognition, natural language processing, and speech synthesis. Key architectures include CNNs, RNNs, and Transformers.

Natural language processing (NLP) focuses on the interaction between computers and human language. Applications include machine translation, sentiment analysis, and question answering. Modern NLP relies heavily on transformer-based models."""

def fixed_chunk(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if end < len(text):
            last_period = chunk.rfind('.')
            if last_period > chunk_size * 0.5:
                chunk = chunk[:last_period + 1]
                end = start + last_period + 1
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if c]

def recursive_chunk(text, separators=['\n\n', '\n', '. ', ' '], chunk_size=200, overlap=50):
    if len(text) <= chunk_size:
        return [text] if text.strip() else []
    sep = separators[0] if separators else ''
    if not sep:
        return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size - overlap)]

    splits = text.split(sep)
    chunks = []
    current = ''
    for split in splits:
        candidate = current + sep + split if current else split
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current.strip())
            if len(split) > chunk_size:
                sub_chunks = recursive_chunk(split, separators[1:], chunk_size, overlap)
                chunks.extend(sub_chunks)
                current = ''
            else:
                current = split
    if current:
        chunks.append(current.strip())
    return chunks

print('=== Text Chunking Strategies ===')
print(f'Original text: {len(text)} chars, {len(text.split())} words')

fixed_chunks = fixed_chunk(text, chunk_size=200, overlap=50)
recursive_chunks = recursive_chunk(text, chunk_size=200, overlap=50)

print(f'\n--- Fixed-size Chunking (size=200, overlap=50) ---')
for i, c in enumerate(fixed_chunks):
    print(f'  Chunk {i}: {len(c)} chars - "{c[:60]}..."')

print(f'\n--- Recursive Character Chunking ---')
for i, c in enumerate(recursive_chunks):
    print(f'  Chunk {i}: {len(c)} chars - "{c[:60]}..."')

print(f'\nKey: Recursive chunking respects natural boundaries (paragraphs > sentences > words).')
print(f'LangChain RecursiveCharacterTextSplitter uses this approach in production.')
print(f'Overlap prevents losing information at chunk boundaries.')

## 2. 元数据提取与文档结构处理

**目的**：保留文档的结构信息，提升检索精度

**元数据类型**：
- **标题层级**：H1/H2/H3标题，提供上下文
- **来源信息**：URL、文件名、页码
- **时间戳**：创建时间、更新时间
- **自定义标签**：领域、语言、权限等级

**复杂文档处理**：
- **表格**：转为自然语言描述或保留结构化格式
- **代码块**：保留完整函数，不切断
- **图片**：提取alt文本或用多模态模型生成描述
- **列表**：保留层级关系

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional

@dataclass
class Document:
    content: str
    metadata: Dict = field(default_factory=dict)

class DocumentProcessor:
    def __init__(self, chunk_size=300, chunk_overlap=50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def process_markdown(self, text, source=''):
        sections = []
        current_headers = {}
        current_content = []

        for line in text.split('\n'):
            header_match = re.match(r'^(#{1,6})\s+(.+)$', line)
            if header_match:
                if current_content:
                    sections.append({
                        'content': '\n'.join(current_content),
                        'headers': dict(current_headers),
                    })
                    current_content = []
                level = len(header_match.group(1))
                current_headers[f'h{level}'] = header_match.group(2)
                for l in range(level + 1, 7):
                    current_headers.pop(f'h{l}', None)
            else:
                current_content.append(line)

        if current_content:
            sections.append({'content': '\n'.join(current_content), 'headers': dict(current_headers)})

        documents = []
        for section in sections:
            content = section['content'].strip()
            if not content:
                continue
            if len(content) > self.chunk_size:
                chunks = recursive_chunk(content, chunk_size=self.chunk_size, overlap=self.chunk_overlap)
            else:
                chunks = [content]
            for chunk in chunks:
                metadata = dict(section['headers'])
                metadata['source'] = source
                metadata['char_count'] = len(chunk)
                documents.append(Document(content=chunk, metadata=metadata))

        return documents

md_text = """# Machine Learning Guide

## Introduction
Machine learning enables systems to learn from data automatically.

## Deep Learning
### Neural Networks
Deep learning uses multi-layer neural networks for feature extraction.
### Transformers
Transformers use self-attention for processing sequential data efficiently."""

processor = DocumentProcessor(chunk_size=200)
docs = processor.process_markdown(md_text, source='ml_guide.md')

print('=== Document Processing with Metadata ===')
for i, doc in enumerate(docs):
    print(f'\nDoc {i}: {doc.metadata.get("char_count", 0)} chars')
    print(f'  Headers: {doc.metadata}')
    print(f'  Content: "{doc.content[:80]}..."')

print(f'\nKey: Metadata (headers, source) provides context for retrieval.')
print(f'Header hierarchy helps LLM understand document structure.')
print(f'Production systems store metadata alongside vectors in the vector database.')

## 3. 向量化与索引构建

**目的**：将文本转换为向量并构建高效检索索引

**向量化模型选择**：
- **OpenAI text-embedding-3-small**：1536维，通用性好，API调用
- **BGE-large-zh**：1024维，中文优秀，本地部署
- **E5-large-v2**：1024维，多语言，本地部署
- **Cohere embed-v3**：1024维，多语言，API调用

**索引类型选择**：
- **Flat（暴力搜索）**：<10K向量，精确结果
- **IVF（倒排文件）**：>100K向量，O(n/nprobe)搜索
- **HNSW（层级导航小世界图）**：<10M向量，O(log n)搜索，延迟低
- **PQ（乘积量化）**：内存受限场景，4-8x压缩

**向量数据库对比**：
- **FAISS**：库，嵌入式，GPU加速，适合研究
- **Milvus**：分布式服务，适合生产
- **Qdrant**：Rust实现，低延迟，适合生产
- **Chroma**：轻量级，适合原型开发

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

class SimpleEncoder(nn.Module):
    def __init__(self, vocab_size=5000, d_model=128, d_embed=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 4, d_model*2, batch_first=True, dropout=0.1), 2
        )
        self.proj = nn.Linear(d_model, d_embed)

    def forward(self, token_ids):
        h = self.embed(token_ids)
        h = self.encoder(h)
        h = h.mean(dim=1)
        return F.normalize(self.proj(h), dim=-1)

encoder = SimpleEncoder()
encoder.eval()

n_docs = 1000
seq_len = 32
doc_ids = torch.randint(0, 5000, (n_docs, seq_len))
query_ids = torch.randint(0, 5000, (5, seq_len))

print('=== Vectorization & Index Construction ===')

with torch.no_grad():
    doc_embeds = encoder(doc_ids)
    query_embeds = encoder(query_ids)

print(f'Document embeddings: {doc_embeds.shape}')
print(f'Query embeddings: {query_embeds.shape}')
print(f'Index size: {doc_embeds.numel() * 4 / 1e6:.2f} MB')

start = time.time()
scores = query_embeds @ doc_embeds.T
brute_force_time = (time.time() - start) * 1000
top_k_indices = scores.topk(5, dim=-1).indices

print(f'\nBrute-force search: {brute_force_time:.2f}ms for {n_docs} vectors')
print(f'Top-5 for query 0: {top_k_indices[0].tolist()}')

print(f'\n--- Index Scaling Analysis ---')
for n in [1000, 10000, 100000, 1000000, 10000000]:
    brute_time = n * 64 * 4 / 1e9 * 1000
    hnsw_time = 16 * 64 * 4 / 1e9 * 1000
    ivf_time = n / 100 * 64 * 4 / 1e9 * 1000
    index_mem = n * 64 * 4 / 1e9
    print(f'  {n:>10,} vectors: brute={brute_time:.1f}ms, HNSW≈{hnsw_time:.1f}ms, IVF≈{ivf_time:.1f}ms, mem={index_mem:.1f}GB')

print(f'\nKey: HNSW provides O(log n) search with low latency.')
print(f'For >1M vectors, use IVF or HNSW instead of brute force.')
print(f'FAISS on GPU can search 1B vectors in <1ms with IVF+PQ.')